# ACE-Step 1.5 / MusicGen Score Synthesizer (Kaggle GPU)
### AI-Content Generation: The Heritage Short Pipeline — Stage 5: Music Score

This notebook generates an original Ethiopian acoustic heritage score for **"የቤታችን ትውስታ"** (*The Memories of Our Home*).
Per the assignment requirements:
- Generates at least 3 distinct candidates (variations in scale, tempo, and arrangement).
- Uses open weights on borrowed GPUs (< 4GB VRAM required).
- Exports high-fidelity candidate WAV files for auditioning and selection.

**Hardware**: Kaggle GPU (T4 / P100)  
**Output**: `candidate_a.wav`, `candidate_b.wav`, `candidate_c.wav`, `selected_score.wav`

In [ ]:
# 1. Install Audio Generation Dependencies
!pip install -q transformers scipy torch torchaudio soundfile

In [ ]:
# 2. Clone Repository or Load Storyboard Music Specifications
import json, os, time
from pathlib import Path

repo_dir = Path("/kaggle/working/ethiopian-heritage-short")
if not repo_dir.exists():
    !git clone https://github.com/Getachew0557/AI-Tradtional-heritage-Content-Generation.git {repo_dir}

storyboard_path = repo_dir / "storyboard.json"
with open(storyboard_path, "r", encoding="utf-8") as f:
    storyboard = json.load(f)

music_meta = storyboard.get("music_track", {})
print(f"Target Score Title: {music_meta.get('title')}")
print(f"Target Scale: {music_meta.get('scale')}")
print(f"Base Prompt: {music_meta.get('caption_prompt')}")

In [ ]:
# 3. Load Audio Generation Model onto GPU (< 4GB VRAM)
import torch
from transformers import MusicgenForConditionalGeneration, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

model_id = "facebook/musicgen-small"
print(f"Loading model weights from {model_id} ...")
processor = AutoProcessor.from_pretrained(model_id)
model = MusicgenForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16).to(device)
print("✅ Music model loaded into GPU memory!")

In [ ]:
# 4. Generate 3 Candidates with Variations in Scale, Tempo & Mood
import scipy.io.wavfile

output_dir = Path("/kaggle/working/outputs/music")
output_dir.mkdir(parents=True, exist_ok=True)

candidates = [
    {
        "id": "candidate_a",
        "title": "Tizita Nostalgia",
        "scale": "Tizita Minor",
        "bpm": 68,
        "prompt": "Gentle Ethiopian acoustic folk score, meditative Tizita pentatonic scale, solo acoustic krar plucks with delicate washint bamboo flute melodies, soft kebero hand drum heartbeat, nostalgic, peaceful, 68 bpm, broadcast acoustic master, no vocals."
    },
    {
        "id": "candidate_b",
        "title": "Bati Heritage Feast",
        "scale": "Bati Scale",
        "bpm": 78,
        "prompt": "Celebratory Ethiopian traditional acoustic ensemble, lively Bati pentatonic scale, rhythmic krar arpeggios, masinko bowed fiddle accents, syncopated kebero percussion, warm joyful family feast, 78 bpm, no vocals."
    },
    {
        "id": "candidate_c",
        "title": "Ambassel Contemplation",
        "scale": "Ambassel Mode",
        "bpm": 72,
        "prompt": "Deep contemplative Ethiopian highland score, Ambassel minor mode, mystical washint bamboo flute lead, deep acoustic bass drone, intimate grandfather wisdom theme, 72 bpm, no vocals."
    }
]

sampling_rate = model.config.audio_encoder.sampling_rate

for cand in candidates:
    cid = cand["id"]
    print(f"\nGenerating {cid}: '{cand['title']}' ({cand['scale']}, {cand['bpm']} BPM) ...")
    inputs = processor(
        text=[cand["prompt"]],
        padding=True,
        return_tensors="pt",
    ).to(device)
    
    # Generate 30 seconds snippet per candidate
    audio_values = model.generate(**inputs, max_new_tokens=1500)
    
    out_path = output_dir / f"{cid}.wav"
    scipy.io.wavfile.write(
        str(out_path),
        rate=sampling_rate,
        data=audio_values[0, 0].cpu().numpy()
    )
    print(f"  ✅ Saved {out_path.name}")

print("\n🎉 All 3 candidates generated successfully!")

In [ ]:
# 5. Package Music Candidates for Download
!zip -j /kaggle/working/music_candidates.zip /kaggle/working/outputs/music/*.wav
print("Created /kaggle/working/music_candidates.zip!")
print("Download and unzip into your local ethiopian-heritage-short/outputs/music/ directory.")